# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the [FAIR² Mass Spectrometry dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa) using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We'll inspect the dataset for record sets (tables), and display their `@id`, name, and associated fields and columns.

In [ ]:
# Print all record sets and their information
if len(metadata.record_sets) == 0:
    print("No record sets declared in metadata.record_sets. Attempting to access via metadata._json...")
    # Fallback: try to find record sets from the raw JSON metadata
    all_keys = list(metadata._json.keys())
    if 'recordSet' in metadata._json:
        print(f"Found 'recordSet' field with {len(metadata._json['recordSet'])} entries.")
        record_sets_metadata = metadata._json['recordSet']
    else:
        # Sometimes the record sets are nested elsewhere.
        record_sets_metadata = []
else:
    print(f"Found {len(metadata.record_sets)} record set(s) in metadata.record_sets.")
    record_sets_metadata = metadata.record_sets

if record_sets_metadata:
    for i, rs in enumerate(record_sets_metadata):
        rs_id = getattr(rs, '@id', None) or rs.get('@id')
        rs_name = getattr(rs, 'name', None) or rs.get('name')
        print(f"[{i}] RecordSet @id: {rs_id}\n    Name: {rs_name}")
        
        # Fields
        fields = getattr(rs, 'fields', None) or rs.get('field') or rs.get('fields') or []
        print(f"    Fields:")
        for f in fields:
            if hasattr(f, '@id'):
                print(f"        - {f['@id'] if isinstance(f, dict) else f.@id}")
            elif isinstance(f, dict) and '@id' in f:
                print(f"        - {f['@id']}")
            else:
                print(f"        - Unnamed Field: {f}")
        # Columns
        columns = getattr(rs, 'columns', None) or rs.get('column') or rs.get('columns') or []
        print(f"    Columns:")
        for c in columns:
            if hasattr(c, '@id'):
                print(f"        - {c['@id'] if isinstance(c, dict) else c.@id}")
            elif isinstance(c, dict) and '@id' in c:
                print(f"        - {c['@id']}")
            else:
                print(f"        - Unnamed Column: {c}")
else:
    print("No record sets found. This dataset may have its records defined in a different way.")

From exploration above, fill in the record set `@id` values below. The FAIR² schema for this dataset defines several record sets in practice. Based on FAIR² conventions, the main record set is typically `dv:dataset` for the main data table, with fields and columns like protein accession, description, abundance, etc.

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. We reference the record set and field IDs by their `@id` values, following Croissant best practices.

In [ ]:
# Specify the list of record set @ids you want to extract from the dataset
# Replace with the actual record set @ids from the above overview. Here we use 'dv:dataset' as a typical main table id for FAIR² datasets.
record_sets = ['dv:dataset']
dataframes = {}

for record_set_id in record_sets:
    try:
        records = list(dataset.records(record_set=record_set_id))
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded {len(dataframes[record_set_id])} records for record set '{record_set_id}'.")
        print(f"Columns: {dataframes[record_set_id].columns.tolist()}")
    except Exception as e:
        print(f"Could not load records for record_set={record_set_id}: {e}")

# Display first few rows of the main record set
main_rs = record_sets[0]
if main_rs in dataframes:
    dataframes[main_rs].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

In this section, we'll choose a numeric field (e.g., protein abundance, peptide count, or MW) using its column `@id`, filter records, normalize, and group by a categorical field. 

**Note:** Replace `<numeric_field_id>` and `<group_field_id>` with the actual `@id` field names as found above, e.g. 'abundance', 'peptide_counts', 'MW', etc.

In [ ]:
# Example: Filter and normalize a numeric field in the main record set.
main_record_set = 'dv:dataset'

# Choose numeric and grouping fields by their @id in the schema/columns; update these as needed
numeric_field = 'MW'  # e.g. Molecular Weight column @id
group_field = 'sample_id'  # e.g. experimental sample identifier column @id

df = dataframes[main_record_set]
if numeric_field in df.columns:
    # Filter records where MW > 10 (as an example threshold; adjust as appropriate)
    threshold = 10
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered {len(filtered_df)} records with {numeric_field} > {threshold}.")

    # Normalize the numeric field
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group by a categorical field if present
    if group_field in df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
        print(f"Grouped mean of {numeric_field} by {group_field}:")
        print(grouped_df.head())
    else:
        print(f"Grouping field '{group_field}' not found in columns.")
else:
    print(f"Numeric field '{numeric_field}' not in DataFrame columns: {df.columns.tolist()}")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Below is an example using matplotlib and seaborn to visualize the molecular weight (MW) distribution and relationship with another numeric field (e.g. 'abundance'). Edit field names as needed.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of MW
if numeric_field in df.columns:
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_field].dropna(), bins=50, kde=True)
    plt.title(f"Distribution of {numeric_field} (MW)")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()

# Scatterplot: MW vs abundance (if available)
other_numeric = 'abundance'  # Example field; replace with the @id/column of interest
if other_numeric in df.columns and numeric_field in df.columns:
    plt.figure(figsize=(7,5))
    sns.scatterplot(x=df[numeric_field], y=df[other_numeric])
    plt.title(f"{numeric_field} vs {other_numeric}")
    plt.xlabel(numeric_field)
    plt.ylabel(other_numeric)
    plt.show()

## 6. Conclusion
We successfully loaded the [FAIR² Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa), explored available record sets, extracted the main data table, performed basic exploratory analysis, and visualized the molecular weight distribution. This demonstrates how to use the Croissant metadata and `mlcroissant` for robust FAIR and reproducible data exploration.

For further research, consult the [dataset schema](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) and associated publication for details about field definitions and experimental context.